# Personal Loan Prediction Using Decision Tree

This notebook mirrors the final Decision Tree workflow used in the project. It covers:

- baseline Decision Tree training
- SMOTE comparison
- hyperparameter tuning with `GridSearchCV`
- cross-validation stability checks
- ROC-AUC and precision-recall evaluation


## 1. Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.tree import DecisionTreeClassifier, plot_tree

## 2. Load Dataset

In [ ]:
DATASET_PATH = Path('Bank_Personal_Loan_Modelling.csv')
TARGET_COLUMN = 'Personal Loan'

df = pd.read_csv(DATASET_PATH)
df.head()

## 3. Dataset Overview

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nTarget distribution:')
print(df[TARGET_COLUMN].value_counts())

## 4. Preprocessing

In [ ]:
df_model = df.drop(columns=['ID', 'ZIP Code'], errors='ignore')

X = df_model.drop(columns=[TARGET_COLUMN])
y = df_model[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Training samples:', len(X_train))
print('Testing samples :', len(X_test))

## 5. Helper Functions

In [ ]:
def build_model():
    return DecisionTreeClassifier(
        criterion='gini',
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        class_weight='balanced',
    )


def build_metrics(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': report['1']['precision'],
        'Recall': report['1']['recall'],
        'F1 Score': report['1']['f1-score'],
    }


## 6. Baseline Decision Tree

In [ ]:
baseline_model = build_model()
baseline_model.fit(X_train, y_train)

y_train_pred = baseline_model.predict(X_train)
y_test_pred = baseline_model.predict(X_test)

print('Train accuracy  :', round(accuracy_score(y_train, y_train_pred), 4))
print('Test accuracy   :', round(accuracy_score(y_test, y_test_pred), 4))
print('\nTest Confusion Matrix:')
print(confusion_matrix(y_test, y_test_pred))
print('\nTest Classification Report:')
print(classification_report(y_test, y_test_pred, digits=4))

baseline_metrics = build_metrics(y_test, y_test_pred)
baseline_metrics

## 7. Feature Importance and Tree Structure

In [ ]:
feature_importance = pd.Series(
    baseline_model.feature_importances_,
    index=X.columns,
).sort_values(ascending=False)
feature_importance

In [ ]:
plt.figure(figsize=(8, 5))
feature_importance.head(10).sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(22, 12))
plot_tree(
    baseline_model,
    feature_names=X.columns,
    class_names=['No Loan', 'Loan'],
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.title('Decision Tree for Personal Loan Prediction')
plt.tight_layout()
plt.show()

## 8. Train/Test Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, matrix, title in zip(
    axes,
    [confusion_matrix(y_train, y_train_pred), confusion_matrix(y_test, y_test_pred)],
    ['Train Confusion Matrix', 'Test Confusion Matrix'],
):
    image = ax.imshow(matrix, cmap='Blues')
    ax.set_title(title)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['No Loan', 'Loan'])
    ax.set_yticklabels(['No Loan', 'Loan'])
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, matrix[i, j], ha='center', va='center')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 9. Accuracy vs Max Depth

In [ ]:
depths = list(range(1, 16))
train_scores = []
test_scores = []

for depth in depths:
    candidate_model = DecisionTreeClassifier(
        criterion='gini',
        max_depth=depth,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        class_weight='balanced',
    )
    candidate_model.fit(X_train, y_train)
    train_scores.append(candidate_model.score(X_train, y_train))
    test_scores.append(candidate_model.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(depths, train_scores, marker='o', label='Training Accuracy')
plt.plot(depths, test_scores, marker='o', label='Testing Accuracy')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Max Depth')
plt.xticks(depths)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [4, 5, 6, 7, 8, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'class_weight': [None, 'balanced'],
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

tuned_model = grid_search.best_estimator_
tuned_params = grid_search.best_params_
tuned_test_pred = tuned_model.predict(X_test)
tuned_metrics = build_metrics(y_test, tuned_test_pred)

print('Best Parameters:')
print(tuned_params)
print('\nTuned Classification Report:')
print(classification_report(y_test, tuned_test_pred, digits=4))
tuned_metrics

## 11. SMOTE Comparison

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

smote_model = build_model()
smote_model.fit(X_train_smote, y_train_smote)
smote_test_pred = smote_model.predict(X_test)
smote_metrics = build_metrics(y_test, smote_test_pred)

print(classification_report(y_test, smote_test_pred, digits=4))
smote_metrics

In [ ]:
comparison_df = pd.DataFrame(
    {
        'Baseline': baseline_metrics,
        'SMOTE': smote_metrics,
        'Tuned': tuned_metrics,
    }
)
comparison_df

In [ ]:
comparison_df.T.plot(kind='bar', figsize=(10, 5))
plt.ylim(0, 1.05)
plt.title('Baseline vs SMOTE vs Tuned Metrics')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 12. Cross-Validation Stability Checks

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
}

baseline_cv = cross_validate(baseline_model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
tuned_cv = cross_validate(tuned_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary = pd.DataFrame(
    {
        'Baseline Mean': [baseline_cv['test_accuracy'].mean(), baseline_cv['test_precision'].mean(), baseline_cv['test_recall'].mean(), baseline_cv['test_f1'].mean()],
        'Tuned Mean': [tuned_cv['test_accuracy'].mean(), tuned_cv['test_precision'].mean(), tuned_cv['test_recall'].mean(), tuned_cv['test_f1'].mean()],
    },
    index=['Accuracy', 'Precision', 'Recall', 'F1'],
)
cv_summary

In [ ]:
cv_summary.plot(kind='bar', figsize=(10, 5), color=['#4C78A8', '#54A24B'])
plt.ylim(0, 1.05)
plt.ylabel('Cross-Validation Score')
plt.title('Baseline vs Tuned Cross-Validation Summary')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 13. ROC-AUC and Precision-Recall Curves

In [ ]:
tuned_scores = tuned_model.predict_proba(X_test)[:, 1]

print('Tuned ROC-AUC           :', round(roc_auc_score(y_test, tuned_scores), 4))
print('Tuned Average Precision :', round(average_precision_score(y_test, tuned_scores), 4))

In [ ]:
fpr, tpr, _ = roc_curve(y_test, tuned_scores)
precision, recall, _ = precision_recall_curve(y_test, tuned_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, color='#4C78A8', label=f'AUC = {roc_auc_score(y_test, tuned_scores):.3f}')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

axes[1].plot(recall, precision, color='#F58518', label=f'AP = {average_precision_score(y_test, tuned_scores):.3f}')
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout()
plt.show()